# How to Achieve Success by Streaming on Twitch​

By: Annette Martin​, Semen Petrov​, Kunanon Chaojaroenrat​, Ismail Cem Bayramoglu​

## Define success​:

* Number of subscribers​
* Number of followers​
* Earnings​
* Monthly viewers​
* Viewer watch time​
* Sponsors​
* Growth rate​

**Streamer:** Make a decision about streaming content and schedule.​

**Administrator:** analyze past performance and forecast about some game that can increase average viewers. Then recommend the option to the streamer.​

**Moderator:** Introduce new streamers, welcome raiders, ban certain people or languages, and increasing the engagement​

**Sponsor:** Decide which streamer to support based on viewers engagement and growth rate​

In [6]:
# Imports all in one location

import os
import pandas as pd
import kagglehub
import requests

In [7]:
# Helper function
def load_first_data_file(folder_path):
    """
    Load the first CSV, TXT, or XLSX file in the given folder as a Pandas DataFrame.
    """
    for file in os.listdir(folder_path):
        full_path = os.path.join(folder_path, file)

        if file.lower().endswith(".csv"):
            # Try UTF-8 first, then fall back to other encodings
            try:
                return pd.read_csv(full_path)
            except UnicodeDecodeError:
                try:
                    return pd.read_csv(full_path, encoding='latin-1')
                except:
                    return pd.read_csv(full_path, encoding='cp1252')

        elif file.lower().endswith(".txt"):
            try:
                return pd.read_csv(full_path, sep="\t")
            except UnicodeDecodeError:
                try:
                    return pd.read_csv(full_path, sep="\t", encoding='latin-1')
                except:
                    return pd.read_csv(full_path, sep="\t", encoding='cp1252')

        elif file.lower().endswith(".xlsx"):
            return pd.read_excel(full_path)

    raise FileNotFoundError(f"No supported data files found in {folder_path}")

In [8]:
datasets = {
    "streamers": "aayushmishra1512/twitchdata", # https://www.kaggle.com/datasets/aayushmishra1512/twitchdata
    "top_games": "rankirsh/evolution-of-top-games-on-twitch", # https://www.kaggle.com/datasets/rankirsh/evolution-of-top-games-on-twitch
}

dfs = {}

for name, kaggle_path in datasets.items():
    folder_path = kagglehub.dataset_download(kaggle_path)
    dfs[name] = load_first_data_file(folder_path)

    print(f"\n{name.upper()} preview:")
    display(dfs[name].head())

Using Colab cache for faster access to the 'twitchdata' dataset.

STREAMERS preview:


,Channel,Watch time(Minutes),Stream time(minutes),Peak viewers,Average viewers,Followers,Followers gained,Views gained,Partnered,Mature,Language
0,xQcOW,6196161750,215250,222720,27716,3246298,1734810,93036735,True,False,English
1,summit1g,6091677300,211845,310998,25610,5310163,1370184,89705964,True,False,English
2,Gaules,5644590915,515280,387315,10976,1767635,1023779,102611607,True,True,Portuguese
3,ESL_CSGO,3970318140,517740,300575,7714,3944850,703986,106546942,True,False,English
4,Tfue,3671000070,123660,285644,29602,8938903,2068424,78998587,True,False,English


Using Colab cache for faster access to the 'evolution-of-top-games-on-twitch' dataset.

TOP_GAMES preview:


,year,Month,Hours_watched,Avg_viewers,Peak_viewers,Streams,Avg_channels,Games_streamed,Viewer_ratio
0,2016,1,480241904,646355,1275257,7701675,20076,12149,29.08
1,2016,2,441859897,635769,1308032,7038520,20427,12134,28.98
2,2016,3,490669308,660389,1591551,7390957,20271,12234,28.92
3,2016,4,377975447,525696,1775120,6869719,16791,12282,28.80
4,2016,5,449836631,605432,1438962,7535519,19394,12424,28.85


In [9]:
display(dfs["streamers"].tail())
display(dfs["top_games"].tail())

,Channel,Watch time(Minutes),Stream time(minutes),Peak viewers,Average viewers,Followers,Followers gained,Views gained,Partnered,Mature,Language
995,LITkillah,122524635,13560,21359,9104,601927,562691,2162107,True,False,Spanish
996,빅헤드 (bighead033),122523705,153000,3940,793,213212,52289,4399897,True,False,Korean
997,마스카 (newmasca),122452320,217410,6431,567,109068,-4942,3417970,True,False,Korean
998,AndyMilonakis,122311065,104745,10543,1153,547446,109111,3926918,True,False,English
999,Remx,122192850,99180,13788,1205,178553,59432,2049420,True,False,French


,year,Month,Hours_watched,Avg_viewers,Peak_viewers,Streams,Avg_channels,Games_streamed,Viewer_ratio
100,2024,5,1727139900,2324548,4032587,22686570,91550,46363,25.90
101,2024,6,1617653502,2249865,4560410,22209935,92133,46752,24.78
102,2024,7,1660370134,2234683,6872963,23309553,93039,46904,24.24
103,2024,8,1711442710,2303422,4159346,23399616,93682,46965,25.14
104,2024,9,1698990192,2362990,4343770,21730541,90841,45635,26.46


# Moving Kaggle Data to BigQuery

In [10]:
# This has already been uploaded, no need to do it again
'''
from google.colab import auth
from google.cloud import bigquery

print("UPLOAD DATAFRAMES TO BIGQUERY")

# Check what dataframes are available
print(f"\nFound {len(dfs)} dataframe(s) to upload:")
for name, df in dfs.items():
    print(f"  - {name}: {df.shape[0]} rows × {df.shape[1]} columns")

# AUTHENTICATE AND SETUP
print("BigQuery Setup")

# Authenticate
auth.authenticate_user()
print("✓ Authenticated with Google Cloud")

# Get BigQuery details
project_id = input("\nEnter your Google Cloud Project ID: ").strip()
dataset_name = input("Enter BigQuery dataset name (e.g., 'twitch_data'): ").strip().lower().replace(' ', '_').replace('-', '_')

# UPLOAD TO BIGQUERY
try:
    # Initialize BigQuery client
    client = bigquery.Client(project=project_id)

    # Create dataset if it doesn't exist
    dataset_id = f"{project_id}.{dataset_name}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"

    try:
        dataset = client.create_dataset(dataset, exists_ok=True)
        print(f"\n✓ Dataset '{dataset_name}' ready in project '{project_id}'")
    except Exception as e:
        print(f"\n⚠ Dataset creation note: {e}")
        print("Continuing with upload...")

    print("Uploading tables to BigQuery...")

    successful_uploads = 0
    failed_uploads = []

    for table_name, df in dfs.items():
        try:
            # Clean table name (BigQuery requirements)
            clean_table_name = table_name.lower().replace(' ', '_').replace('-', '_')
            table_id = f"{dataset_id}.{clean_table_name}"

            # Clean column names (BigQuery requirements)
            # Remove special characters and replace with underscores
            df_clean = df.copy()
            df_clean.columns = [
                col.replace('(', '_')
                   .replace(')', '_')
                   .replace(' ', '_')
                   .replace('-', '_')
                   .replace('.', '_')
                   .replace('/', '_')
                   .replace('\\', '_')
                   .replace(',', '_')
                   .replace(';', '_')
                   .replace(':', '_')
                   .replace('!', '_')
                   .replace('?', '_')
                   .replace('#', '_')
                   .replace('&', '_')
                   .replace('%', '_')
                   .replace('*', '_')
                   .replace('+', '_')
                   .replace('=', '_')
                   .replace('<', '_')
                   .replace('>', '_')
                   .replace('[', '_')
                   .replace(']', '_')
                   .replace('{', '_')
                   .replace('}', '_')
                   .replace('|', '_')
                   .replace('~', '_')
                   .replace('`', '_')
                   .replace('"', '_')
                   .replace("'", '_')
                   .strip('_')  # Remove leading/trailing underscores
                for col in df.columns
            ]

            # Remove consecutive underscores
            df_clean.columns = [
                '_'.join(filter(None, col.split('_')))
                for col in df_clean.columns
            ]

            # Configure the upload
            job_config = bigquery.LoadJobConfig(
                write_disposition="WRITE_TRUNCATE",  # Overwrite if exists
                autodetect=True  # Auto-detect schema
            )

            # Upload the dataframe
            print(f"Uploading {clean_table_name}...", end=" ")
            job = client.load_table_from_dataframe(
                df_clean, table_id, job_config=job_config
            )
            job.result()  # Wait for completion

            print(f"✓ ({df.shape[0]} rows)")
            successful_uploads += 1

        except Exception as e:
            print(f"✗")
            print(f"  Error: {e}")
            failed_uploads.append(table_name)

    print(f"\n✓ Successfully uploaded {successful_uploads}/{len(dfs)} table(s)")

    if failed_uploads:
        print(f"\n⚠ Failed uploads: {', '.join(failed_uploads)}")

    print(f"\nYour data is now in BigQuery:")
    print(f"  Project: {project_id}")
    print(f"  Dataset: {dataset_name}")
    print(f"  Tables: {', '.join([name for name in dfs.keys()])}")
    print(f"\nView your data at:")
    print(f"  https://console.cloud.google.com/bigquery?project={project_id}")
    print(f"\nExample queries:")
    for name in dfs.keys():
        clean_name = name.lower().replace(' ', '_').replace('-', '_')
        print(f"  SELECT * FROM `{project_id}.{dataset_name}.{clean_name}` LIMIT 10;")
    print("=" * 80)

except Exception as e:
    print(f"\n✗ Error uploading to BigQuery: {e}")
    print("\nTroubleshooting:")
    print("  1. Verify your project exists at: https://console.cloud.google.com/")
    print("  2. Enable BigQuery API: https://console.cloud.google.com/apis/library/bigquery.googleapis.com")
    print("  3. Check your Project ID is correct (use hyphens, not underscores)")
    print(f"  4. Current Project ID attempt: '{project_id}'")
    '''

'\nfrom google.colab import auth\nfrom google.cloud import bigquery\n\nprint("UPLOAD DATAFRAMES TO BIGQUERY")\n\n# Check what dataframes are available\nprint(f"\nFound {len(dfs)} dataframe(s) to upload:")\nfor name, df in dfs.items():\n    print(f"  - {name}: {df.shape[0]} rows × {df.shape[1]} columns")\n\n# AUTHENTICATE AND SETUP\nprint("BigQuery Setup")\n\n# Authenticate\nauth.authenticate_user()\nprint("✓ Authenticated with Google Cloud")\n\n# Get BigQuery details\nproject_id = input("\nEnter your Google Cloud Project ID: ").strip()\ndataset_name = input("Enter BigQuery dataset name (e.g., \'twitch_data\'): ").strip().lower().replace(\' \', \'_\').replace(\'-\', \'_\')\n\n# UPLOAD TO BIGQUERY\ntry:\n    # Initialize BigQuery client\n    client = bigquery.Client(project=project_id)\n\n    # Create dataset if it doesn\'t exist\n    dataset_id = f"{project_id}.{dataset_name}"\n    dataset = bigquery.Dataset(dataset_id)\n    dataset.location = "US"\n\n    try:\n        dataset = client.

# Twitch direct information

In [11]:
# If we want more metrics outside of Kaggle, here is infor from:
  # https://dev.twitch.tv/docs/api/
  # I made the account under my email, but I think it should run for everyone

# Credentials
CLIENT_ID = 'bzzl4wywab65nscubsjrpfy22vlsbs'
CLIENT_SECRET = 'jl3yczpr3j5qrs1qdsfjgfx6fgx4fx'

# OAuth token
auth_url = 'https://id.twitch.tv/oauth2/token'
auth_params = {
    'client_id': CLIENT_ID,
    'client_secret': CLIENT_SECRET,
    'grant_type': 'client_credentials'
}

response = requests.post(auth_url, params=auth_params)
access_token = response.json()['access_token']

# Set up headers for API requests
headers = {
    'Client-ID': CLIENT_ID,
    'Authorization': f'Bearer {access_token}'
}

In [12]:
# Example: Get top streams
streams_url = 'https://api.twitch.tv/helix/streams'
response = requests.get(streams_url, headers=headers, params={'first': 20})
streams_data = response.json()

display(streams_data)

{'data': [{'id': '316691137120',
   'user_id': '22510310',
   'user_login': 'gamesdonequick',
   'user_name': 'GamesDoneQuick',
   'game_id': '10322',
   'game_name': 'FINAL FANTASY X',
   'type': 'live',
   'title': 'AGDQ 2026 benefiting Prevent Cancer Foundation - Final Fantasy X !donate !schedule',
   'viewer_count': 32956,
   'started_at': '2026-01-04T16:02:44Z',
   'language': 'en',
   'thumbnail_url': 'https://static-cdn.jtvnw.net/previews-ttv/live_user_gamesdonequick-{width}x{height}.jpg',
   'tag_ids': [],
   'tags': ['English', 'Speedrun', 'Charity'],
   'is_mature': False},
  {'id': '316269860695',
   'user_id': '220476955',
   'user_login': 'ishowspeed',
   'user_name': 'IShowSpeed',
   'game_id': '509672',
   'game_name': 'IRL',
   'type': 'live',
   'title': 'irl Safari stream in Okavango Delta🐘🦒🦁(Botswana)',
   'viewer_count': 28779,
   'started_at': '2026-01-05T11:49:13Z',
   'language': 'en',
   'thumbnail_url': 'https://static-cdn.jtvnw.net/previews-ttv/live_user_ishow

In [13]:
def get_comprehensive_data(username):

    # Basic user info
    user_url = 'https://api.twitch.tv/helix/users'
    user_response = requests.get(user_url, headers=headers, params={'login': username})

    if not user_response.json()['data']:
        print(f"User {username} not found")
        return None

    user_data = user_response.json()['data'][0]
    user_id = user_data['id']

    # Follower count
    followers_url = 'https://api.twitch.tv/helix/channels/followers'
    followers_response = requests.get(followers_url, headers=headers, params={'broadcaster_id': user_id})
    follower_count = followers_response.json()['total']

    # Check if currently live and get viewer count
    streams_url = 'https://api.twitch.tv/helix/streams'
    streams_response = requests.get(streams_url, headers=headers, params={'user_id': user_id})
    streams_data = streams_response.json()['data']

    is_live = len(streams_data) > 0
    current_viewers = streams_data[0]['viewer_count'] if is_live else 0
    current_game = streams_data[0]['game_name'] if is_live else 'Offline'
    stream_title = streams_data[0]['title'] if is_live else 'N/A'

    # Recent videos/VODs
    videos_url = 'https://api.twitch.tv/helix/videos'
    videos_response = requests.get(videos_url, headers=headers, params={
        'user_id': user_id,
        'first': 10,
        'type': 'archive'  # Past broadcasts
    })
    videos_data = videos_response.json()['data']

    # Calculate average views from recent VODs
    recent_vod_views = [video['view_count'] for video in videos_data]
    avg_vod_views = sum(recent_vod_views) / len(recent_vod_views) if recent_vod_views else 0

    # Channel info
    channel_url = 'https://api.twitch.tv/helix/channels'
    channel_response = requests.get(channel_url, headers=headers, params={'broadcaster_id': user_id})
    channel_data = channel_response.json()['data'][0]

    return {
        'username': username,
        'display_name': user_data['display_name'],
        'user_id': user_id,
        'followers': follower_count,
        'is_live': is_live,
        'current_viewers': current_viewers,
        'current_game': current_game,
        'stream_title': stream_title,
        'avg_vod_views': round(avg_vod_views, 0),
        'recent_streams': len(videos_data),
        'account_created': user_data['created_at'],
        'broadcaster_type': user_data['broadcaster_type'],  # 'partner', 'affiliate', or ''
        'description': user_data['description']
    }

def get_stream_history(username, max_streams=20):
    """Get detailed history of recent streams"""

    # User ID first
    user_url = 'https://api.twitch.tv/helix/users'
    user_response = requests.get(user_url, headers=headers, params={'login': username})
    user_id = user_response.json()['data'][0]['id']

    # Videos (past broadcasts)
    videos_url = 'https://api.twitch.tv/helix/videos'
    videos_response = requests.get(videos_url, headers=headers, params={
        'user_id': user_id,
        'first': max_streams,
        'type': 'archive'
    })
    videos = videos_response.json()['data']

    stream_history = []
    for video in videos:
        stream_history.append({
            'title': video['title'],
            'created_at': video['created_at'],
            'view_count': video['view_count'],
            'duration': video['duration'],
            'url': video['url']
        })

    return pd.DataFrame(stream_history)

# Data for multiple streamers
streamers = ['ninja', 'pokimane', 'xqc', 'shroud']  # Replace with your target streamers

print("Fetching data for streamers...")
data = []
for streamer in streamers:
    print(f"Getting data for {streamer}...")
    streamer_data = get_comprehensive_data(streamer)
    if streamer_data:
        data.append(streamer_data)

# Create main DataFrame
df = pd.DataFrame(data)

print("\n STREAMER METRICS")
print(df[['display_name', 'followers', 'is_live', 'current_viewers', 'avg_vod_views', 'broadcaster_type']])

# Detailed stream history for one streamer
print("\n STREAM HISTORY FOR FIRST STREAMER")
if streamers:
    history_df = get_stream_history(streamers[0])
    print(history_df)

Fetching data for streamers...
Getting data for ninja...
Getting data for pokimane...
Getting data for xqc...
Getting data for shroud...

 STREAMER METRICS
  display_name  followers  is_live  current_viewers  avg_vod_views  \
0        Ninja   19260584    False                0       239568.0   
1     pokimane    9389681    False                0       316643.0   
2          xQc   12247882    False                0       689041.0   
3       shroud   11318441    False                0       401714.0   

  broadcaster_type  
0          partner  
1          partner  
2          partner  
3          partner  

 STREAM HISTORY FOR FIRST STREAMER
                                                title            created_at  \
0          [DROPS ON] LAST STREAM TILL MONDAY/TUESDAY  2026-01-02T15:05:39Z   
1              [DROPS ON] FIRST STREAM OF THE YEAR :D  2026-01-01T13:20:03Z   
2   [DROPS ON] SHORT ISH STREAM TODAY FAMILY IS ST...  2025-12-30T14:01:02Z   
3                         [DROPS ON]